# ToneFit ML — Kaggle Pipeline

**FaRL vs DINOv2 vs MCF for Personal Color Season Classification**  
Dataset: Deep Armocromia (Stacchio et al., ECCV 2024)

---

**Before running:**
1. Set accelerator to **GPU P100** (Settings → Accelerator → GPU P100)
2. Add the RGB-M dataset (Settings → Add Data → your uploaded RGB-M dataset)
3. Make sure your latest code is pushed to GitHub before cloning

**Pipeline:**
1. Check GPU
2. Clone repo + install dependencies
3. Link RGB-M dataset from Kaggle input
4. Preprocessing — extract CIELab/HSV features for EDA
5. EDA — visualize dataset
6. Train FaRL (Model A)
7. Train DINOv2 (Model B)
8. Train MCF (Model C)
9. Evaluate — compare all three models
10. Save results

---
## Step 0 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU available: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌ No GPU detected!')
    print('Go to Settings → Accelerator → GPU P100, then re-run.')

---
## Step 1 — Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/ajipal/ToneFit.git'
REPO_DIR = '/kaggle/working/ToneFit'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned. Pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'✅ Working directory: {os.getcwd()}')
!ls

In [ ]:
!pip install -q timm>=0.9.0 scikit-image imagehash
print('✅ Dependencies installed')

---
## Step 2 — Link RGB-M Dataset from Kaggle Input

> Update `KAGGLE_DATASET_SLUG` to match the name of your uploaded Kaggle dataset.
> The dataset should contain the `RGB-M/` folder at its root.

In [ ]:
import os

# ── UPDATE THIS to match your Kaggle dataset slug ──
KAGGLE_DATASET_SLUG = 'deep-armocromia-rgb-m'   # e.g. /kaggle/input/deep-armocromia-rgb-m/

KAGGLE_INPUT_PATH = f'/kaggle/input/{KAGGLE_DATASET_SLUG}/RGB-M'
LOCAL_LINK        = '/kaggle/working/ToneFit/RGB-M'

if not os.path.exists(LOCAL_LINK):
    os.symlink(KAGGLE_INPUT_PATH, LOCAL_LINK)
    print(f'✅ Linked: {LOCAL_LINK} → {KAGGLE_INPUT_PATH}')
else:
    print(f'✅ RGB-M already linked')

# Verify structure
print('\nDataset structure:')
for split in ['train', 'test']:
    split_path = os.path.join(LOCAL_LINK, split)
    if os.path.exists(split_path):
        seasons = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
        total = 0
        print(f'  {split}/')
        for season in sorted(seasons):
            count = sum(len(files) for _, _, files in os.walk(os.path.join(split_path, season)))
            total += count
            print(f'    {season}/: {count} images')
        print(f'    TOTAL: {total} images')
    else:
        print(f'  ❌ {split}/ not found — check KAGGLE_DATASET_SLUG above')

---
## Step 3 — Preprocessing (EDA Features Only)

> Extracts CIELab/HSV features from training images for EDA visualization.  
> Does NOT re-split the dataset — RGB-M/train/ and RGB-M/test/ are used as-is.

In [ ]:
import sys
if 'preprocess' in sys.modules:
    del sys.modules['preprocess']

import preprocess
preprocess.run()
print('✅ Preprocessing complete')

In [ ]:
import pandas as pd
df = pd.read_csv('features.csv')
print(f'features.csv: {len(df)} rows')
print('\nClass distribution:')
print(df['season'].value_counts())
print('\nSample:')
df.head()

---
## Step 4 — EDA

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

os.makedirs('results', exist_ok=True)
df = pd.read_csv('features.csv')

SEASONS = ['autumn', 'spring', 'summer', 'winter']
COLORS  = {'autumn': '#B7410E', 'spring': '#F4A261', 'summer': '#90B4CE', 'winter': '#4169E1'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ToneFit ML — EDA Overview', fontsize=14, fontweight='bold')

# Class distribution
counts = df['season'].value_counts()[SEASONS]
axes[0].bar(counts.index, counts.values, color=[COLORS[s] for s in counts.index])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Images')
for i, (s, v) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# ITA Score by season
data = [df[df['season'] == s]['ITA'].values for s in SEASONS]
bp = axes[1].boxplot(data, labels=[s.capitalize() for s in SEASONS], patch_artist=True)
for patch, s in zip(bp['boxes'], SEASONS):
    patch.set_facecolor(COLORS[s])
    patch.set_alpha(0.7)
axes[1].set_title('ITA Score by Season\n(Higher = Lighter/Cooler)')
axes[1].set_ylabel('ITA Score')

# a* vs b* scatter (undertone)
for s in SEASONS:
    sub = df[df['season'] == s]
    axes[2].scatter(sub['a_mean'], sub['b_mean'], label=s.capitalize(),
                    color=COLORS[s], alpha=0.5, s=20)
axes[2].axhline(0, color='gray', ls='--', lw=0.8)
axes[2].axvline(0, color='gray', ls='--', lw=0.8)
axes[2].set_title('a* vs b* (Undertone)')
axes[2].set_xlabel('a* (warm/cool)')
axes[2].set_ylabel('b* (yellow/blue)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA saved to results/eda_overview.png')

---
## Step 5 — Train FaRL (Model A)

> FaRL pretrained weights are downloaded automatically from GitHub releases (~650 MB).  
> If weights are not found, automatically falls back to ResNeXt50.  
> Expected training time: ~30–45 minutes on P100 GPU

In [ ]:
os.makedirs('models', exist_ok=True)

!wget -q --show-progress -O models/farl_weights.pth \
  "https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep64.pth"

print(f'✅ Done: {os.path.getsize("models/farl_weights.pth")/1e6:.0f} MB')

In [ ]:
import sys, importlib

if 'train_farl' in sys.modules:
    importlib.reload(sys.modules['train_farl'])
else:
    import train_farl

train_farl.main()
print('✅ FaRL training complete')

In [ ]:
from IPython.display import Image as IPImage, display
if os.path.exists('results/farl_training.png'):
    display(IPImage('results/farl_training.png'))
else:
    print('Training curves not found yet')

---
## Step 6 — Train DINOv2 (Model B)

> DINOv2 weights are downloaded automatically via timm (~330 MB).  
> Last 2 transformer blocks are unfrozen for task-specific fine-tuning.  
> Expected training time: ~1.5–2.5 hours on P100 GPU

In [ ]:
import sys, importlib

if 'train_dinov2' in sys.modules:
    importlib.reload(sys.modules['train_dinov2'])
else:
    import train_dinov2

train_dinov2.main()
print('✅ DINOv2 training complete')

In [ ]:
from IPython.display import Image as IPImage, display
if os.path.exists('results/dinov2_training.png'):
    display(IPImage('results/dinov2_training.png'))
else:
    print('Training curves not found yet')

---
## Step 7 — Evaluate All Models

> Run AFTER all three models are trained.  
> Compares FaRL vs DINOv2 vs MCF against paper baselines from Stacchio et al. (2024)

In [ ]:
os.makedirs('models', exist_ok=True)

# Download MCF pretrained weights from Google Drive using gdown
!pip install -q gdown
!gdown --id 1hZqMKUfc2J6zGMDLPB4m32fIzi0TSecU -O models/mcf_weights.pth

if os.path.exists('models/mcf_weights.pth'):
    print(f'✅ MCF weights: {os.path.getsize("models/mcf_weights.pth")/1e6:.0f} MB')
else:
    print('⚠️  Download failed — MCF will fall back to ImageNet-pretrained ViT-B/16')

In [ ]:
import sys, importlib

if 'train_mcf' in sys.modules:
    importlib.reload(sys.modules['train_mcf'])
else:
    import train_mcf

train_mcf.main()
print('✅ MCF training complete')

In [ ]:
from IPython.display import Image as IPImage, display
if os.path.exists('results/mcf_training.png'):
    display(IPImage('results/mcf_training.png'))
else:
    print('Training curves not found yet')

---
## Step 7 — Evaluate Both Models

> Run AFTER both models are trained.  
> Compares FaRL vs DINOv2 against paper baselines from Stacchio et al. (2024)

In [ ]:
import zipfile, glob

zip_path = '/kaggle/working/ToneFit_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('results/*'):
        z.write(f)
    for f in glob.glob('models/*.pth'):
        if 'farl_weights' not in f and 'mcf_weights' not in f:   # skip large pretrained weights
            z.write(f)
    if os.path.exists('features.csv'):
        z.write('features.csv')

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'✅ Saved: {zip_path} ({size_mb:.1f} MB)')
print('Download from: Kaggle notebook → Output panel → ToneFit_results.zip')

In [ ]:
import pandas as pd
if os.path.exists('results/comparison_table.csv'):
    df_results = pd.read_csv('results/comparison_table.csv')
    print('\n===== MODEL COMPARISON =====')
    print(df_results.to_string(index=False))
else:
    print('comparison_table.csv not found')

In [ ]:
from IPython.display import Image as IPImage, display
for img in ['results/confusion_farl.png', 'results/confusion_dinov2.png', 'results/model_comparison.png']:
    if os.path.exists(img):
        print(f'\n{img}:')
        display(IPImage(img))
    else:
        print(f'❌ {img} not found')

---
## Step 8 — Save Results

> All files saved to `/kaggle/working/ToneFit/` are automatically available  
> for download from the Kaggle notebook output panel after the session ends.

In [ ]:
import zipfile, glob

zip_path = '/kaggle/working/ToneFit_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('results/*'):
        z.write(f)
    for f in glob.glob('models/*.pth'):
        if 'farl_weights' not in f:   # skip the 650 MB pretrained weights
            z.write(f)
    if os.path.exists('features.csv'):
        z.write('features.csv')

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'✅ Saved: {zip_path} ({size_mb:.1f} MB)')
print('Download from: Kaggle notebook → Output panel → ToneFit_results.zip')